# TRABALHO AUTÓNOMO 2

O objetivo deste trabalho é completar o programa do agente para explorar o mundo Wumpus tentando obter a máxima performance antes de o agente morrer ou sair da caverna com o ouro e eventualmente matar o Wumpus


Primeiro vamos importar os pacotes necessários:

In [2]:
from ipythonblocks import BlockGrid
from agents import *
from logic4e import *

## Coisas que existem no mundo

Depois vamos definir as coisas que existem no mundo Wumpus

In [3]:
class Ouro(Thing):
    pass

class Parede(Thing):
    pass

class Brilho(Thing):
    pass

class Poco(Thing):
    pass

class Brisa(Thing):
    pass

class Flecha(Thing):
    pass

class Wumpus(Agent):
    gritou = False
    pass

class Fedor(Thing):
    pass

class Grito(Agent):
    pass

## O Ambiente

Vamos defenir a classe do ambiente:

In [4]:
class WumpusEnvironment(XYEnvironment):

    probabilidade = 0.1  # probabilidade de espalhar poços pelo ambiente

    ''' A caverna quando for de 4x4 deve ter mais 2 extra para as paredes'''
    def __init__(self, agente, dim=4):
        super().__init__(dim+2, dim+2)
        self.init_world(agente)

    
    def init_world(self,agente):
        '''Espalha coisas pela caverna de acordo com a probabilidade definida'''

        ''' atualiza começo e final'''

        "Paredes"
        
        for x in range(self.width-1):
            self.add_thing(Parede(), (x, 0),True)
            self.add_thing(Parede(), (x, self.height - 1),True)
        for y in range(0, self.height):
            self.add_thing(Parede(), (0, y),True)
            self.add_thing(Parede(), (self.width - 1, y),True)

        self.x_start, self.y_start = (1, 1)
        self.x_end, self.y_end = (self.width - 1, self.height - 1)

        "Poços"
        for x in range(self.x_start, self.x_end):
            for y in range(self.y_start, self.y_end):
                if random.random() < self.probabilidade:
                    self.add_thing(Poco(), (x, y), True)
                    self.add_thing(Brisa(), (x - 1, y), True)
                    self.add_thing(Brisa(), (x, y - 1), True)
                    self.add_thing(Brisa(), (x + 1, y), True)
                    self.add_thing(Brisa(), (x, y + 1), True)

        "Wumpus"
        w_x, w_y = self.random_location_inbounds(exclude=(1, 1))
        self.add_thing(Wumpus(lambda x: ""), (w_x, w_y), True)
        self.add_thing(Fedor(), (w_x - 1, w_y), True)
        self.add_thing(Fedor(), (w_x + 1, w_y), True)
        self.add_thing(Fedor(), (w_x, w_y - 1), True)
        self.add_thing(Fedor(), (w_x, w_y + 1), True)

        "Ouro"
        self.add_thing(Ouro(), self.random_location_inbounds(exclude=(1, 1)), True)

        "Agente"
        self.add_thing(agente, (1, 1), True)

    def get_world(self, show_walls=True):
        """Devolve as coisas que existem no mundo"""
        result = []
        x_start, y_start = (0, 0) if show_walls else (1, 1)
        
        if show_walls:
            x_end, y_end = self.width, self.height
        else:
            x_end, y_end = self.width - 1, self.height - 1

        for y in range(y_start, y_end):
            row = []
            for x in range(x_start, x_end):
                row.append(self.list_things_at((x, y)))
            result.append(row)
        return result

    def percepts_from(self, agent, location, tclass=Thing):
        """ Devolve as perceções para cada localização definida e atualiza alguns items com as perceções"""
        thing_percepts = {
            Ouro: Brilho(),
            Parede: Bump(),
            Wumpus: Fedor(),
            Poco: Brisa()}

        """ O agente não é perceção de si próprio"""
        thing_percepts[agent.__class__] = None

        """ O ouro no quarto brilha"""
        if location != agent.location:
            thing_percepts[Ouro] = None

        '''junta as perceções'''
        result = [thing_percepts.get(thing.__class__, thing) for thing in self.things
                  if thing.location == location and isinstance(thing, tclass)]
        
        return result if len(result) else [None]

    def percept(self, agent):
        ''' retorna as coisas nos quartos adjacentes (não diagonais)
        no formato : [quarto à esquerda, quarto à direita, quarto em cima, quarto em baixo, na localização atual]'''
        (x, y) = agent.location
        result = []
        result.append(self.percepts_from(agent, (x - 1, y)))
        result.append(self.percepts_from(agent, (x + 1, y)))
        result.append(self.percepts_from(agent, (x, y - 1)))
        result.append(self.percepts_from(agent, (x, y + 1)))
        result.append(self.percepts_from(agent, (x, y)))

        '''Quando o wumpos morre ele grita'''
        wumpus = [thing for thing in self.things if isinstance(thing, Wumpus)]
        if len(wumpus) and not wumpus[0].alive and not wumpus[0].gritou:
            result[-1].append(Grito())
            wumpus[0].gritou = True

        return result

    def execute_action(self, agent, action):
        """Modifica o estado do ambiente com base nas ações do agente.
        Atualiza a performance do agente"""

        if isinstance(agent, Explorador) and self.in_danger(agent):
            return
            
        agent.bump = False
        if action in ['direita', 'esquerda', 'frente', 'pega']:
            agent.executa_acao(action)
            if action == 'frente' :
                self.add_thing(agent,agent.location,True)
            agent.performance -= 1
        elif action == 'sair':
            if agent.location == (1, 1):  # Agente só pode sair no quarto (1,1)
                agent.performance += 1000 if Ouro() in agent.holding else 0
                self.delete_thing(agent)
        elif action == 'dispara':
            """A flecha atravessa diretamente na direção corrente do agente"""
            if agent.has_arrow:
                local_flecha = agent.quadrado_a_seguir('frente', agent.location)
                while self.is_inbounds(local_flecha):
                    wumpus = [thing for thing in self.list_things_at(local_flecha) if isinstance(thing, Wumpus)]
                    if len(wumpus):
                        wumpus[0].alive = False
                        break
                    local_flecha = agent.quadrado_a_seguir('frente', local_flecha)
                agent.has_arrow = False

    def in_danger(self, agent):
        """Verifica se o agente está em perigo, se sim o agente morre"""
        for thing in self.list_things_at(agent.location):
            if isinstance(thing, Poco) or (isinstance(thing, Wumpus) and thing.alive):
                agent.alive = False
                agent.performance -= 1000
                agent.morto_por = thing.__class__.__name__
                return True
        return False

    def is_done(self):
        """O Jogo finaliza se o agente é morto ou se sobe no quarto [1,1]"""
        explorador = [agent for agent in self.agents if isinstance(agent, Explorador)]
        if len(explorador):
            if explorador[0].alive:
                return False
            else:
                print("Morto por {} [-1000].".format(explorador[0].morto_por))
        else:
            print("O Explorador subiu em {}."
                  .format("como ouro [+1000]!" if Ouro() not in self.things else "sem o ouro [+0]"))
        return True
    
    def step(self):
        agent = [thing for thing in self.things if isinstance(thing, Explorador)][0]
        for action in agent.programa(self.percept(agent)):
            self.execute_action(agent, action)


# Agente e o seu programa

E aqui definimos o próprio agente e o seu programa

In [5]:
class Explorador(Thing):

    def __init__(self, dimencao):
        self.dimrow = dimencao
        self.alive = True
        self.bump = False
        self.holding = []
        self.performance = 0
        self.tem_flecha = True
        self.morto_por = ""
        self.t = 0
        self.location = (1,1)
        self.direcao = 0 # 0: cima, 1: direita, 2: baixo, 3: esquerda

    def quadrado_a_seguir(self,local):
        if self.direcao == 0 :
            return (local[0],local[1] + 1)
        if self.direcao == 1 :
            return (local[0] + 1,local[1])
        if self.direcao == 2 :
            return (local[0],local[1] - 1)
        if self.direcao == 3 :
            return (local[0] - 1,local[1])
        
    def atualiza_loc_dir(self, acao):
        if acao == 'frente' :
            self.location = self.quadrado_a_seguir(self.location)
        if acao == 'esquerda' :
            self.direcao = (self.direcao - 1 ) % 4
        if acao == 'direita' :
            self.direcao = (self.direcao + 1) % 4 

    def executa_acao(self,acao):
        self.atualiza_loc_dir(acao)
        pass
      

    def programa(self,percecao):
        print('Perceção [esquerda, direita, baixo, cima, aqui] = {}; ação? '.format(percecao))
        acao = ['direita','frente']
        return acao


## Criação do agente e do mundo

Vamos então criar o mundo e po-lo a correr no tempo.

In [6]:
random.seed(1234567)

global grid, w
dimensao = 10
explorador = Explorador(dimensao)
w = WumpusEnvironment(explorador, dimensao)
grid = BlockGrid(w.width, w.height, fill=(0, 0, 0))

def draw_grid(world):
    color = {"Brisa": (128, 128, 128),
        "Poco": (128,128,128),
        "Ouro": (255, 255, 0),
        "Brilho": (255, 255, 0),
        "Wumpus": (255, 0, 0),
        "Fedor": (255, 0, 0),
        "Explorador": (0, 0, 255),
        "Parede": (255, 255, 255)
        }
    grid[:] = (0, 0, 0)
    for y in range(0,len(world)):
        for x in range(0,len(world[y])):
            if len(world[y][x]):
                grid[w.height-1-y, x] = color[world[y][x][-1].__class__.__name__]


clear_output(1)
draw_grid(w.get_world(True))
grid.show()
w.step()
sleep(2)
clear_output(1)
draw_grid(w.get_world(True))
grid.show()
print('A performance do explorador foi: {}'.format(explorador.performance))

,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,
,,,,,,,,,,,


A performance do explorador foi: -2
